# Freshness/diversity weight tuning

Owner: Chua Rui Ning, Renee

This notebook searches for reranker settings using the validation split. It tunes only the freshness bonus weight and MMR diversity lambda. The freshness grace period stays at 0 hours and half-life stays at 24 hours, matching the current reranker defaults.

To reduce runtime, the grid search uses a reproducible 10% sample of validation users, preloads the data once, caches article-pair similarity calculations across trials, and tests only a small fixed grid. It then reranks and evaluates the selected setting once on the full validation split. Do not treat the sampled search as final evidence; the full-validation confirmation is still validation, and final reporting should use the frozen settings on test once.

The selection rule is explicit: keep settings within the configured Recall@5 and nDCG@5 loss limits for every configured model, then maximize a weighted average of headroom-normalized gains in freshness share and shared intra-list diversity. Change the limits or objective weights only to values agreed by the group before running.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
EVALUATION_CODE_DIR = PROJECT_ROOT / 'Evaluation' / 'code'
VALIDATION_LABELS_PATH = PROJECT_ROOT / 'feature_engineering' / 'validation_features.parquet'
ARTICLES_PATH = PROJECT_ROOT / 'Evaluation' / 'catalogue' / 'article_union.parquet'
SIMILARITY_CONSTANTS_PATH = (
    PROJECT_ROOT / 'Evaluation' / 'evaluation_outputs' / 'stage2' / 'validation'
    / 'diversity_constants.json'
)
OUTPUT_DIR = PROJECT_ROOT / 'Evaluation' / 'Stage 2 - Reranking' / 'weights'

required_code = ('evaluate.py', 'rerank.py', 'similarity.py', 'optimize_reranker.py')
missing_code = [name for name in required_code if not (EVALUATION_CODE_DIR / name).is_file()]
if missing_code:
    raise FileNotFoundError(
        f'Missing {missing_code} under {EVALUATION_CODE_DIR}. '
        'Copy the updated Evaluation support-code folder to Drive first.'
    )
sys.path.insert(0, str(EVALUATION_CODE_DIR))
from optimize_reranker import optimize_reranker_weights

required_inputs = [VALIDATION_LABELS_PATH, ARTICLES_PATH, SIMILARITY_CONSTANTS_PATH]
missing_inputs = [str(path) for path in required_inputs if not Path(path).is_file()]
if missing_inputs:
    raise FileNotFoundError(
        'Required optimizer input(s) not found:\n' + '\n'.join(missing_inputs)
        + '\nRun the shared evaluator/catalogue build and Stage 2 diversity-constants cell first.'
    )

# Optimize for the selected finalist(s), not every experiment. The result is
# one shared parameter pair that must satisfy the relevance guard for each.
MODEL_RUNS = [
    # {
    #     'name': 'LSTM',
    #     'rankings': PROJECT_ROOT / 'Experiments' / '2.1_lstm' / '2.1.3_improved_run' / '20261104_175920_improved_seed42' / 'rankings' / 'validation' / 'submission_top_200.parquet',
    # },
    {
        'name': 'RRF LSTM75 + XGBoost25',
        'rankings': PROJECT_ROOT / 'Evaluation' / 'ensemble' / 'validation' / 'rrf_lstm75_xgboost25_validation_top_200.parquet',
    }
]

print('Labels:', VALIDATION_LABELS_PATH, VALIDATION_LABELS_PATH.exists())
print('Catalogue:', ARTICLES_PATH, ARTICLES_PATH.exists())
print('Similarity constants:', SIMILARITY_CONSTANTS_PATH, SIMILARITY_CONSTANTS_PATH.exists())
for run in MODEL_RUNS:
    print(run['name'], run['rankings'], Path(run['rankings']).exists())

Labels: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/validation_features.parquet True
Catalogue: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet True
Similarity constants: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/evaluation_outputs/stage2/validation/diversity_constants.json True
RRF LSTM75 + XGBoost25 /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/ensemble/validation/rrf_lstm75_xgboost25_validation_top_200.parquet True


## Search settings

These are proposed defaults, not a group-approved relevance tolerance. The absolute and relative caps both apply; the stricter cap is used. The no-op (freshness weight 0, lambda 1) is always included as the reference. Half-life and grace period are held fixed to keep this search small.

In [3]:
FRESHNESS_WEIGHT_GRID = [0.0, 0.1, 0.25, 0.5, 0.75]
MMR_LAMBDA_GRID = [0.6, 0.8, 0.9, 1.0]

FRESHNESS_GRACE_H = 0.0
FRESHNESS_HALF_LIFE_H = 24.0
OUTPUT_K = 5
TUNE_USER_FRACTION = 0.20
TUNE_SEED = 42

# Provisional relevance guards: effective permitted loss is the stricter of
# the relative limit and absolute 0.02-point cap. Confirm these with the group.
MAX_RELATIVE_RECALL5_DROP = 0.10
MAX_ABSOLUTE_RECALL5_DROP = 0.02
MAX_RELATIVE_NDCG5_DROP = 0.10
MAX_ABSOLUTE_NDCG5_DROP = 0.02

# Equal objective weights across freshness and diversity, after normalizing by
# each metric's remaining headroom to 1.0.
FRESHNESS_OBJECTIVE_WEIGHT = 0.5
DIVERSITY_OBJECTIVE_WEIGHT = 0.5

candidate_count = len({(float(w), float(lam)) for w in FRESHNESS_WEIGHT_GRID for lam in MMR_LAMBDA_GRID} | {(0.0, 1.0)})
print('Candidate configurations:', candidate_count)
print('User sample fraction:', TUNE_USER_FRACTION)
print('Models tuned together:', len(MODEL_RUNS))

Candidate configurations: 20
User sample fraction: 0.2
Models tuned together: 1


## Run the small search and full-validation confirmation

The full grid metrics table and selected settings appear below. The grid table is saved as reranker_weight_search_sample.csv. Full-validation base-versus-selected metrics and guard results are displayed in the notebook only; no top-five ranking or per-request files are written.

In [4]:
result = optimize_reranker_weights(
    validation_labels_path=VALIDATION_LABELS_PATH,
    articles_path=ARTICLES_PATH,
    similarity_constants_path=SIMILARITY_CONSTANTS_PATH,
    model_runs=MODEL_RUNS,
    output_dir=OUTPUT_DIR,
    freshness_weights=FRESHNESS_WEIGHT_GRID,
    diversity_lambdas=MMR_LAMBDA_GRID,
    freshness_grace_h=FRESHNESS_GRACE_H,
    freshness_half_life_h=FRESHNESS_HALF_LIFE_H,
    output_k=OUTPUT_K,
    sample_user_fraction=TUNE_USER_FRACTION,
    sample_seed=TUNE_SEED,
    max_relative_recall5_drop=MAX_RELATIVE_RECALL5_DROP,
    max_absolute_recall5_drop=MAX_ABSOLUTE_RECALL5_DROP,
    max_relative_ndcg5_drop=MAX_RELATIVE_NDCG5_DROP,
    max_absolute_ndcg5_drop=MAX_ABSOLUTE_NDCG5_DROP,
    freshness_objective_weight=FRESHNESS_OBJECTIVE_WEIGHT,
    diversity_objective_weight=DIVERSITY_OBJECTIVE_WEIGHT,
)

# Display all tried settings with all tracked metrics (not just the first 10).
display(result['sample_trials'])
display(result['full_validation_summary'])
print('Metrics table and selected settings saved under:', result['output_dir'])

Tuned config 1/20: freshness_weight=0.0, lambda=0.6
Tuned config 2/20: freshness_weight=0.0, lambda=0.8
Tuned config 3/20: freshness_weight=0.0, lambda=0.9
Tuned config 4/20: freshness_weight=0.0, lambda=1.0
Tuned config 5/20: freshness_weight=0.1, lambda=0.6
Tuned config 6/20: freshness_weight=0.1, lambda=0.8
Tuned config 7/20: freshness_weight=0.1, lambda=0.9
Tuned config 8/20: freshness_weight=0.1, lambda=1.0
Tuned config 9/20: freshness_weight=0.25, lambda=0.6
Tuned config 10/20: freshness_weight=0.25, lambda=0.8
Tuned config 11/20: freshness_weight=0.25, lambda=0.9
Tuned config 12/20: freshness_weight=0.25, lambda=1.0
Tuned config 13/20: freshness_weight=0.5, lambda=0.6
Tuned config 14/20: freshness_weight=0.5, lambda=0.8
Tuned config 15/20: freshness_weight=0.5, lambda=0.9
Tuned config 16/20: freshness_weight=0.5, lambda=1.0
Tuned config 17/20: freshness_weight=0.75, lambda=0.6
Tuned config 18/20: freshness_weight=0.75, lambda=0.8
Tuned config 19/20: freshness_weight=0.75, lambda

,freshness_weight,diversity_lambda,sample_requests,sample_users,rrf_lstm75_xgboost25_known_click_recall@5,rrf_lstm75_xgboost25_observed_click_ndcg@5,rrf_lstm75_xgboost25_age_le_24h_share@5,rrf_lstm75_xgboost25_shared_intra_list_diversity@5,rrf_lstm75_xgboost25_freshness_headroom_gain,rrf_lstm75_xgboost25_diversity_headroom_gain,relevance_guard_pass,objective
0,0.00,0.9,6071,344,0.195333,0.125316,1.0,0.950556,0.0,0.783706,True,0.783706
1,0.10,0.9,6071,344,0.192203,0.123420,1.0,0.949083,0.0,0.777264,True,0.777264
2,0.25,1.0,6071,344,0.201488,0.125121,1.0,0.795775,0.0,0.106618,True,0.106618
3,0.10,1.0,6071,344,0.207006,0.132039,1.0,0.781334,0.0,0.043446,True,0.043446
4,0.00,1.0,6071,344,0.211256,0.134882,1.0,0.771403,0.0,0.000000,True,0.000000
5,0.00,0.8,6071,344,0.178395,0.117880,1.0,0.980519,0.0,0.914781,False,NaN
6,0.00,0.6,6071,344,0.170241,0.114243,1.0,0.996049,0.0,0.982715,False,NaN
7,0.10,0.8,6071,344,0.179850,0.117905,1.0,0.979348,0.0,0.909658,False,NaN
8,0.10,0.6,6071,344,0.171312,0.113984,1.0,0.995547,0.0,0.980520,False,NaN
9,0.25,0.9,6071,344,0.181661,0.114956,1.0,0.947367,0.0,0.769757,False,NaN


,model,stage,requests,label_requests_available,requests_without_export,requests_with_known_clicks,requests_without_known_clicks,users,logged_candidate_rows,ranked_catalogue_rows,...,median_article_age_h@5,unknown_age_share@5,age_le_24h_share@5,age_24_72h_share@5,age_gt_72h_share@5,topic_coverage@5,topic_article_coverage@5,intra_list_diversity@5,shared_intra_list_diversity@5,distinct_categories@5
0,RRF LSTM75 + XGBoost25,base_top5,29110,29110,0,29110,0,1719,344666,145550,...,1.620654,0.0,1.0,0.0,0.0,11.734387,1.0,0.769057,0.769057,2.492099
1,RRF LSTM75 + XGBoost25,optimized_top5,29110,29110,0,29110,0,1719,344666,145550,...,1.469521,0.0,1.0,0.0,0.0,15.275919,1.0,0.950682,0.950682,3.392271


Metrics table and selected settings saved under: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/weights


# Reading the result

- Paste the parameter values printed above into the Stage 2 reranker. Continuity remains off.
- The sampled grid results, including all tracked metrics for every setting, are saved as reranker_weight_search_sample.csv.
- The selected parameter JSON and full-validation relevance-guard outcome are saved. Full-validation base-versus-selected metrics are displayed above but are not exported as another CSV.
- No optimized top-five rankings or per-request Parquet files are written by this optimizer.
- If the full-validation relevance guard fails even though the sample search passed, do not treat the selected weights as guard-approved.
- These results are validation-tuned. Freeze the final model and weights before evaluating the labelled test split once.

Verdict:
- Freshness weight = 0.00 (Due to LSTM accounting for freshness)
- Diversity lambda = 0.10
